# PARTE 3 — Modelos No Lineales
## Proyecto de Analítica de Datos – Pádel

**Objetivo:** Entrenar modelos no lineales (Random Forest y SVM) para predecir la probabilidad de que un equipo gane un partido de pádel tras cada punto.

**Salidas requeridas:**
- `modelos_no_lineales.pkl`
- `reporte_modelos_no_lineales.pdf`

---
## 0. Instalación de Dependencias

In [ ]:
# Ejecutar solo si faltan dependencias
# !pip install scikit-learn xgboost pandas numpy matplotlib seaborn joblib fpdf2 imbalanced-learn

---
## 1. Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
import warnings
import os
from datetime import datetime

# Modelos
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Preprocesamiento
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Validación y métricas
from sklearn.model_selection import (
    StratifiedKFold, cross_validate, GridSearchCV, RandomizedSearchCV
)
from sklearn.metrics import (
    roc_auc_score, log_loss, brier_score_loss,
    accuracy_score, classification_report,
    roc_curve, confusion_matrix, ConfusionMatrixDisplay
)

# PDF
from fpdf import FPDF

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('✅ Librerías importadas correctamente')
print(f'   Fecha de ejecución: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

---
## 2. Carga del Dataset Final

> **Instrucción:** Carga el archivo generado en la Parte 1 (`dataframe_final.csv` o `dataframe_final.pkl`).

In [ ]:
# ─── AJUSTA LA RUTA SEGÚN TU ENTORNO ───────────────────────────────────────
DATA_PATH = 'dataframe_final.csv'   # o 'dataframe_final.pkl'
TARGET_COL = 'gano_equipo1'         # 1 = Equipo 1 ganó el punto / partido
# ────────────────────────────────────────────────────────────────────────────

if DATA_PATH.endswith('.pkl'):
    df = pd.read_pickle(DATA_PATH)
else:
    df = pd.read_csv(DATA_PATH)

print(f'Shape del dataset: {df.shape}')
print(f'\nDistribución del target "{TARGET_COL}":')
print(df[TARGET_COL].value_counts(normalize=True).round(4))
df.head()

---
## 3. Preparación de Features

In [ ]:
# ─── COLUMNAS A EXCLUIR (IDs, texto, leakage) ───────────────────────────────
EXCLUDE_COLS = [
    TARGET_COL,
    'partido_id', 'punto_id', 'jugador_id',
    'timestamp', 'frame', 'video_id',
    'nombre', 'equipo'
]
EXCLUDE_COLS = [c for c in EXCLUDE_COLS if c in df.columns]
# ─────────────────────────────────────────────────────────────────────────────

feature_cols = [c for c in df.columns if c not in EXCLUDE_COLS]

# Codificar columnas categóricas
cat_cols = df[feature_cols].select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

X = df[feature_cols].copy()
y = df[TARGET_COL].copy()

print(f'Features seleccionadas: {X.shape[1]}')
print(f'Muestras: {X.shape[0]}')
print(f'\nClases en target: {y.unique()}')
if cat_cols:
    print(f'Columnas categóricas codificadas: {cat_cols}')

---
## 4. Configuración de Validación Cruzada

In [ ]:
CV_FOLDS   = 5
RANDOM_STATE = 42

cv_strategy = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

SCORING = {
    'roc_auc'  : 'roc_auc',
    'neg_log_loss' : 'neg_log_loss',
    'neg_brier': 'neg_brier_score',
    'accuracy' : 'accuracy'
}

print(f'✅ Validación cruzada estratificada configurada: {CV_FOLDS} folds')

---
## 5. Modelo 1 — Random Forest
### 5.1 Modelo Base (sin tuning)

In [ ]:
rf_base = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_base_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', rf_base)
])

cv_rf_base = cross_validate(
    rf_base_pipe, X, y,
    cv=cv_strategy,
    scoring=SCORING,
    return_train_score=True
)

print('Random Forest — Modelo Base')
print(f'  AUC      : {cv_rf_base["test_roc_auc"].mean():.4f} ± {cv_rf_base["test_roc_auc"].std():.4f}')
print(f'  Log Loss : {-cv_rf_base["test_neg_log_loss"].mean():.4f} ± {cv_rf_base["test_neg_log_loss"].std():.4f}')
print(f'  Brier    : {-cv_rf_base["test_neg_brier"].mean():.4f} ± {cv_rf_base["test_neg_brier"].std():.4f}')
print(f'  Accuracy : {cv_rf_base["test_accuracy"].mean():.4f} ± {cv_rf_base["test_accuracy"].std():.4f}')

### 5.2 Hyperparameter Tuning — Random Forest (RandomizedSearchCV)

In [ ]:
rf_param_grid = {
    'model__n_estimators'      : [100, 200, 300, 500],
    'model__max_depth'         : [None, 5, 10, 15, 20],
    'model__min_samples_split' : [2, 5, 10],
    'model__min_samples_leaf'  : [1, 2, 4],
    'model__max_features'      : ['sqrt', 'log2', 0.5],
    'model__class_weight'      : [None, 'balanced']
}

rf_pipe_tune = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

rf_search = RandomizedSearchCV(
    rf_pipe_tune,
    param_distributions=rf_param_grid,
    n_iter=30,
    scoring='roc_auc',
    cv=cv_strategy,
    refit=True,
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_search.fit(X, y)

print(f'\n✅ Mejor AUC (CV): {rf_search.best_score_:.4f}')
print(f'Mejores hiperparámetros:\n{rf_search.best_params_}')

### 5.3 Evaluación del Mejor Random Forest

In [ ]:
rf_best_pipe = rf_search.best_estimator_

cv_rf_best = cross_validate(
    rf_best_pipe, X, y,
    cv=cv_strategy,
    scoring=SCORING,
    return_train_score=True
)

rf_results = {
    'Modelo'   : 'Random Forest (tuned)',
    'AUC'      : cv_rf_best['test_roc_auc'].mean(),
    'AUC_std'  : cv_rf_best['test_roc_auc'].std(),
    'LogLoss'  : -cv_rf_best['test_neg_log_loss'].mean(),
    'Brier'    : -cv_rf_best['test_neg_brier'].mean(),
    'Accuracy' : cv_rf_best['test_accuracy'].mean(),
    'AUC_train': cv_rf_best['train_roc_auc'].mean()
}

print('Random Forest — Modelo Tuneado')
for k, v in rf_results.items():
    if k != 'Modelo':
        print(f'  {k:<12}: {v:.4f}')

### 5.4 Importancia de Variables — Random Forest

In [ ]:
# Entrenar en el dataset completo para extraer importancias
rf_best_pipe.fit(X, y)
rf_model = rf_best_pipe.named_steps['model']

feat_imp = pd.Series(
    rf_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

top_n = min(20, len(feat_imp))

fig, ax = plt.subplots(figsize=(9, 6))
feat_imp.head(top_n).plot(kind='barh', ax=ax, color='steelblue')
ax.invert_yaxis()
ax.set_title(f'Top {top_n} Features — Random Forest', fontsize=14, fontweight='bold')
ax.set_xlabel('Importancia (Mean Decrease Impurity)')
plt.tight_layout()
plt.savefig('rf_feature_importance.png', bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado: rf_feature_importance.png')

---
## 6. Modelo 2 — SVM (Clasificación)
### 6.1 Modelo Base (sin tuning)

In [ ]:
svm_base_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', SVC(probability=True, random_state=RANDOM_STATE))
])

cv_svm_base = cross_validate(
    svm_base_pipe, X, y,
    cv=cv_strategy,
    scoring=SCORING,
    return_train_score=True
)

print('SVM — Modelo Base')
print(f'  AUC      : {cv_svm_base["test_roc_auc"].mean():.4f} ± {cv_svm_base["test_roc_auc"].std():.4f}')
print(f'  Log Loss : {-cv_svm_base["test_neg_log_loss"].mean():.4f} ± {cv_svm_base["test_neg_log_loss"].std():.4f}')
print(f'  Brier    : {-cv_svm_base["test_neg_brier"].mean():.4f} ± {cv_svm_base["test_neg_brier"].std():.4f}')
print(f'  Accuracy : {cv_svm_base["test_accuracy"].mean():.4f} ± {cv_svm_base["test_accuracy"].std():.4f}')

### 6.2 Hyperparameter Tuning — SVM (GridSearchCV)

In [ ]:
svm_param_grid = {
    'model__C'      : [0.01, 0.1, 1, 10, 100],
    'model__kernel' : ['rbf', 'linear'],
    'model__gamma'  : ['scale', 'auto'],
    'model__class_weight': [None, 'balanced']
}

svm_pipe_tune = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', SVC(probability=True, random_state=RANDOM_STATE))
])

svm_search = GridSearchCV(
    svm_pipe_tune,
    param_grid=svm_param_grid,
    scoring='roc_auc',
    cv=cv_strategy,
    refit=True,
    verbose=1,
    n_jobs=-1
)

svm_search.fit(X, y)

print(f'\n✅ Mejor AUC (CV): {svm_search.best_score_:.4f}')
print(f'Mejores hiperparámetros:\n{svm_search.best_params_}')

### 6.3 Evaluación del Mejor SVM

In [ ]:
svm_best_pipe = svm_search.best_estimator_

cv_svm_best = cross_validate(
    svm_best_pipe, X, y,
    cv=cv_strategy,
    scoring=SCORING,
    return_train_score=True
)

svm_results = {
    'Modelo'   : 'SVM (tuned)',
    'AUC'      : cv_svm_best['test_roc_auc'].mean(),
    'AUC_std'  : cv_svm_best['test_roc_auc'].std(),
    'LogLoss'  : -cv_svm_best['test_neg_log_loss'].mean(),
    'Brier'    : -cv_svm_best['test_neg_brier'].mean(),
    'Accuracy' : cv_svm_best['test_accuracy'].mean(),
    'AUC_train': cv_svm_best['train_roc_auc'].mean()
}

print('SVM — Modelo Tuneado')
for k, v in svm_results.items():
    if k != 'Modelo':
        print(f'  {k:<12}: {v:.4f}')

---
## 7. Comparación Global de Modelos

In [ ]:
comparison_data = [
    {
        'Modelo'   : 'Random Forest (base)',
        'AUC'      : cv_rf_base['test_roc_auc'].mean(),
        'LogLoss'  : -cv_rf_base['test_neg_log_loss'].mean(),
        'Brier'    : -cv_rf_base['test_neg_brier'].mean(),
        'Accuracy' : cv_rf_base['test_accuracy'].mean(),
    },
    {
        'Modelo'   : 'Random Forest (tuned)',
        'AUC'      : rf_results['AUC'],
        'LogLoss'  : rf_results['LogLoss'],
        'Brier'    : rf_results['Brier'],
        'Accuracy' : rf_results['Accuracy'],
    },
    {
        'Modelo'   : 'SVM (base)',
        'AUC'      : cv_svm_base['test_roc_auc'].mean(),
        'LogLoss'  : -cv_svm_base['test_neg_log_loss'].mean(),
        'Brier'    : -cv_svm_base['test_neg_brier'].mean(),
        'Accuracy' : cv_svm_base['test_accuracy'].mean(),
    },
    {
        'Modelo'   : 'SVM (tuned)',
        'AUC'      : svm_results['AUC'],
        'LogLoss'  : svm_results['LogLoss'],
        'Brier'    : svm_results['Brier'],
        'Accuracy' : svm_results['Accuracy'],
    },
]

comp_df = pd.DataFrame(comparison_data).set_index('Modelo')
comp_df = comp_df.round(4)
print(comp_df.to_string())

In [ ]:
# Gráfico comparativo de AUC
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# AUC
comp_df['AUC'].plot(
    kind='bar', ax=axes[0],
    color=['#4C72B0', '#55A868', '#C44E52', '#DD8452'],
    edgecolor='white'
)
axes[0].set_title('AUC por Modelo', fontsize=13, fontweight='bold')
axes[0].set_ylabel('AUC')
axes[0].set_ylim(max(0, comp_df['AUC'].min() - 0.05), 1.0)
axes[0].tick_params(axis='x', rotation=30)
axes[0].axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Baseline (0.5)')
axes[0].legend()

# Log Loss
comp_df['LogLoss'].plot(
    kind='bar', ax=axes[1],
    color=['#4C72B0', '#55A868', '#C44E52', '#DD8452'],
    edgecolor='white'
)
axes[1].set_title('Log Loss por Modelo (menor = mejor)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Log Loss')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Comparación de Modelos No Lineales — Pádel', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('comparacion_modelos_no_lineales.png', bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado: comparacion_modelos_no_lineales.png')

---
## 8. Curvas ROC Comparativas

In [ ]:
from sklearn.model_selection import StratifiedKFold

fig, ax = plt.subplots(figsize=(8, 6))

models_to_plot = [
    ('Random Forest (tuned)', rf_best_pipe,  '#55A868'),
    ('SVM (tuned)',           svm_best_pipe, '#C44E52'),
]

for name, pipe, color in models_to_plot:
    tprs, aucs = [], []
    mean_fpr = np.linspace(0, 1, 100)
    for train_idx, test_idx in cv_strategy.split(X, y):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
        pipe.fit(X_tr, y_tr)
        proba = pipe.predict_proba(X_te)[:, 1]
        fpr, tpr, _ = roc_curve(y_te, proba)
        tprs.append(np.interp(mean_fpr, fpr, tpr))
        aucs.append(roc_auc_score(y_te, proba))
    mean_tpr = np.mean(tprs, axis=0)
    ax.plot(mean_fpr, mean_tpr, label=f'{name} (AUC={np.mean(aucs):.3f})', color=color, lw=2)

ax.plot([0,1],[0,1], 'k--', label='Baseline')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Curvas ROC (Media CV) — Modelos No Lineales', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('roc_curves_no_lineales.png', bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado: roc_curves_no_lineales.png')

---
## 9. Matriz de Confusión (Mejor Modelo)

In [ ]:
from sklearn.model_selection import cross_val_predict

# Seleccionar el mejor modelo globalmente
best_auc_rf  = rf_results['AUC']
best_auc_svm = svm_results['AUC']

if best_auc_rf >= best_auc_svm:
    best_model_name = 'Random Forest (tuned)'
    best_model_pipe = rf_best_pipe
else:
    best_model_name = 'SVM (tuned)'
    best_model_pipe = svm_best_pipe

print(f'Mejor modelo global: {best_model_name}')

y_pred_cv = cross_val_predict(best_model_pipe, X, y, cv=cv_strategy)

fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y, y_pred_cv)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Matriz de Confusión — {best_model_name}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix_best.png', bbox_inches='tight')
plt.show()

print('\nReporte de Clasificación:')
print(classification_report(y, y_pred_cv))

---
## 10. Guardado de Modelos

In [ ]:
# Reentrenar en el dataset completo antes de guardar
rf_best_pipe.fit(X, y)
svm_best_pipe.fit(X, y)

modelos_no_lineales = {
    'random_forest' : rf_best_pipe,
    'svm'           : svm_best_pipe,
    'best_model'    : best_model_pipe,
    'best_model_name': best_model_name,
    'feature_cols'  : feature_cols,
    'resultados_cv' : {
        'rf_tuned' : rf_results,
        'svm_tuned': svm_results
    },
    'metadata': {
        'fecha': datetime.now().isoformat(),
        'cv_folds': CV_FOLDS,
        'random_state': RANDOM_STATE,
        'n_features': len(feature_cols),
        'n_samples': len(y)
    }
}

joblib.dump(modelos_no_lineales, 'modelos_no_lineales.pkl')
print('✅ Modelos guardados en: modelos_no_lineales.pkl')

# Verificar tamaño
size_mb = os.path.getsize('modelos_no_lineales.pkl') / 1e6
print(f'   Tamaño: {size_mb:.2f} MB')

---
## 11. Generación del Reporte PDF

In [ ]:
class ReportePDF(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 11)
        self.set_fill_color(30, 80, 150)
        self.set_text_color(255, 255, 255)
        self.cell(0, 10, 'Proyecto Analítica Pádel — Parte 3: Modelos No Lineales', 0, 1, 'C', fill=True)
        self.ln(2)

    def footer(self):
        self.set_y(-12)
        self.set_font('Helvetica', 'I', 8)
        self.set_text_color(128)
        self.cell(0, 8, f'Página {self.page_no()} | Generado: {datetime.now().strftime("%Y-%m-%d %H:%M")}', 0, 0, 'C')

    def section_title(self, title):
        self.set_font('Helvetica', 'B', 12)
        self.set_fill_color(220, 230, 245)
        self.set_text_color(20, 50, 100)
        self.cell(0, 8, title, 0, 1, 'L', fill=True)
        self.ln(1)

    def body_text(self, text):
        self.set_font('Helvetica', '', 10)
        self.set_text_color(40, 40, 40)
        self.multi_cell(0, 6, text)
        self.ln(1)

    def metric_row(self, label, value, highlight=False):
        self.set_font('Helvetica', 'B' if highlight else '', 10)
        if highlight:
            self.set_fill_color(200, 230, 200)
            self.set_text_color(0, 100, 0)
        else:
            self.set_fill_color(245, 245, 245)
            self.set_text_color(50, 50, 50)
        self.cell(90, 7, f'  {label}', 0, 0, 'L', fill=True)
        self.cell(90, 7, str(value), 0, 1, 'R', fill=True)
        self.ln(0.5)

    def add_image_safe(self, path, w=180):
        if os.path.exists(path):
            self.image(path, x=15, w=w)
            self.ln(4)
        else:
            self.body_text(f'[Imagen no disponible: {path}]')


pdf = ReportePDF()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_page()

# ── Portada ────────────────────────────────────────────────────────────────
pdf.set_font('Helvetica', 'B', 18)
pdf.set_text_color(20, 50, 120)
pdf.ln(8)
pdf.cell(0, 12, 'REPORTE — MODELOS NO LINEALES', 0, 1, 'C')
pdf.set_font('Helvetica', '', 12)
pdf.set_text_color(80, 80, 80)
pdf.cell(0, 8, 'Random Forest  &  SVM (Clasificación)', 0, 1, 'C')
pdf.cell(0, 8, f'Generado: {datetime.now().strftime("%Y-%m-%d %H:%M")}', 0, 1, 'C')
pdf.ln(6)

# ── Introducción ────────────────────────────────────────────────────────────
pdf.section_title('1. Objetivo')
pdf.body_text(
    'Este reporte documenta el entrenamiento, tuning y evaluación de dos modelos '
    'no lineales para predecir la probabilidad de que un equipo gane un partido de '
    'pádel tras cada punto: Random Forest y SVM (clasificación).\n'
    'Se utilizó validación cruzada estratificada de 5 folds y las métricas '
    'AUC, Log Loss, Brier Score y Accuracy.'
)

# ── Dataset ──────────────────────────────────────────────────────────────────
pdf.section_title('2. Dataset')
pdf.metric_row('Muestras totales', len(y))
pdf.metric_row('Features utilizadas', len(feature_cols))
pdf.metric_row('Variable objetivo', TARGET_COL)
pdf.metric_row('Folds de validación cruzada', CV_FOLDS)
pdf.ln(3)

# ── Random Forest ─────────────────────────────────────────────────────────────
pdf.section_title('3. Random Forest')
pdf.body_text(
    'Se entrenó un Random Forest base y se realizó tuning con RandomizedSearchCV '
    '(30 iteraciones) sobre los hiperparámetros: n_estimators, max_depth, '
    'min_samples_split, min_samples_leaf, max_features y class_weight.'
)
pdf.body_text('Resultados del mejor modelo (tuned):')
pdf.metric_row('AUC (media CV)',      f"{rf_results['AUC']:.4f} ± {rf_results['AUC_std']:.4f}", highlight=True)
pdf.metric_row('Log Loss (media CV)', f"{rf_results['LogLoss']:.4f}")
pdf.metric_row('Brier Score',         f"{rf_results['Brier']:.4f}")
pdf.metric_row('Accuracy',            f"{rf_results['Accuracy']:.4f}")
pdf.metric_row('AUC Train',           f"{rf_results['AUC_train']:.4f}")
pdf.ln(3)

# Mejores hiperparámetros RF
pdf.body_text('Mejores hiperparámetros (RandomizedSearchCV):')
for k, v in rf_search.best_params_.items():
    pdf.metric_row(k.replace('model__', ''), str(v))
pdf.ln(3)

# Imagen importancia features
pdf.body_text('Importancia de variables (Top 20):')
pdf.add_image_safe('rf_feature_importance.png', w=175)

# ── SVM ───────────────────────────────────────────────────────────────────────
pdf.add_page()
pdf.section_title('4. SVM (Support Vector Machine)')
pdf.body_text(
    'Se entrenó un SVM con kernel RBF como base, aplicando escalado estándar '
    '(StandardScaler). El tuning se realizó con GridSearchCV sobre los '
    'hiperparámetros: C, kernel, gamma y class_weight.'
)
pdf.body_text('Resultados del mejor modelo (tuned):')
pdf.metric_row('AUC (media CV)',      f"{svm_results['AUC']:.4f} ± {svm_results['AUC_std']:.4f}", highlight=True)
pdf.metric_row('Log Loss (media CV)', f"{svm_results['LogLoss']:.4f}")
pdf.metric_row('Brier Score',         f"{svm_results['Brier']:.4f}")
pdf.metric_row('Accuracy',            f"{svm_results['Accuracy']:.4f}")
pdf.metric_row('AUC Train',           f"{svm_results['AUC_train']:.4f}")
pdf.ln(3)

pdf.body_text('Mejores hiperparámetros (GridSearchCV):')
for k, v in svm_search.best_params_.items():
    pdf.metric_row(k.replace('model__', ''), str(v))
pdf.ln(3)

# ── Comparación ───────────────────────────────────────────────────────────────
pdf.section_title('5. Comparación de Modelos')
pdf.add_image_safe('comparacion_modelos_no_lineales.png', w=175)

# Tabla comparativa
pdf.set_font('Helvetica', 'B', 10)
pdf.set_fill_color(30, 80, 150)
pdf.set_text_color(255, 255, 255)
cols = ['Modelo', 'AUC', 'LogLoss', 'Brier', 'Accuracy']
widths = [60, 32, 32, 32, 32]
for col, w in zip(cols, widths):
    pdf.cell(w, 7, col, 1, 0, 'C', fill=True)
pdf.ln()

pdf.set_font('Helvetica', '', 9)
for i, row in comp_df.reset_index().iterrows():
    pdf.set_fill_color(245 if i % 2 == 0 else 255, 245 if i % 2 == 0 else 255, 245 if i % 2 == 0 else 255)
    pdf.set_text_color(40, 40, 40)
    pdf.cell(60, 6, str(row['Modelo']), 1, 0, 'L', fill=True)
    pdf.cell(32, 6, f"{row['AUC']:.4f}",      1, 0, 'C', fill=True)
    pdf.cell(32, 6, f"{row['LogLoss']:.4f}",  1, 0, 'C', fill=True)
    pdf.cell(32, 6, f"{row['Brier']:.4f}",    1, 0, 'C', fill=True)
    pdf.cell(32, 6, f"{row['Accuracy']:.4f}", 1, 1, 'C', fill=True)
pdf.ln(4)

# ── Curvas ROC ────────────────────────────────────────────────────────────────
pdf.add_page()
pdf.section_title('6. Curvas ROC')
pdf.add_image_safe('roc_curves_no_lineales.png', w=160)

# ── Matriz de confusión ───────────────────────────────────────────────────────
pdf.section_title('7. Matriz de Confusión — Mejor Modelo')
pdf.body_text(f'Mejor modelo global seleccionado: {best_model_name}')
pdf.add_image_safe('confusion_matrix_best.png', w=120)

# ── Conclusiones ──────────────────────────────────────────────────────────────
pdf.section_title('8. Conclusiones')
pdf.body_text(
    f'• Mejor modelo global: {best_model_name}\n'
    f'• AUC Random Forest (tuned): {rf_results["AUC"]:.4f}\n'
    f'• AUC SVM (tuned):           {svm_results["AUC"]:.4f}\n'
    '\n'
    'El Random Forest destaca por su capacidad de capturar relaciones no lineales '
    'entre las variables del partido y por la interpretabilidad a través de la '
    'importancia de features. El SVM ofrece un buen desempeño en espacios de alta '
    'dimensionalidad, siendo robusto ante overfitting con el parámetro C bien calibrado.\n'
    '\n'
    'Los modelos entrenados en esta parte servirán como referencia para la '
    'comparación con XGBoost (Parte 4).'
)

# ── Artefactos generados ──────────────────────────────────────────────────────
pdf.section_title('9. Artefactos Generados')
pdf.metric_row('modelos_no_lineales.pkl',               'Pipelines RF y SVM con mejores hiperparámetros')
pdf.metric_row('reporte_modelos_no_lineales.pdf',        'Este reporte')
pdf.metric_row('rf_feature_importance.png',              'Importancia de variables RF')
pdf.metric_row('comparacion_modelos_no_lineales.png',    'Gráfico comparativo AUC / LogLoss')
pdf.metric_row('roc_curves_no_lineales.png',             'Curvas ROC comparativas')
pdf.metric_row('confusion_matrix_best.png',              'Matriz de confusión mejor modelo')

pdf.output('reporte_modelos_no_lineales.pdf')
print('✅ Reporte PDF generado: reporte_modelos_no_lineales.pdf')

---
## 12. Resumen Final

In [ ]:
print('=' * 55)
print('  PARTE 3 — RESUMEN FINAL')
print('=' * 55)
print(f'  Random Forest (tuned)  → AUC: {rf_results["AUC"]:.4f}  |  LogLoss: {rf_results["LogLoss"]:.4f}')
print(f'  SVM (tuned)            → AUC: {svm_results["AUC"]:.4f}  |  LogLoss: {svm_results["LogLoss"]:.4f}')
print(f'  Mejor modelo global    → {best_model_name}')
print('─' * 55)
print('  Archivos generados:')
print('    ✅ modelos_no_lineales.pkl')
print('    ✅ reporte_modelos_no_lineales.pdf')
print('=' * 55)